# OSEMN: il percorso di un progetto di data science

Un progetto basato sui dati non comincia da un algoritmo, ma da una sequenza di attività. Un modo semplice per ricordarle è **OSEMN**, pronunciato come la parola inglese *awesome*:

1. **O — Obtain:** ottenere i dati;
2. **S — Scrub:** pulire e preparare i dati;
3. **E — Explore:** esplorare i dati;
4. **M — Model:** costruire e valutare un modello;
5. **N — iNterpret:** interpretare e comunicare i risultati.

Le fasi non sono sempre lineari: un problema scoperto durante l'esplorazione può richiedere una nuova pulizia o perfino una nuova raccolta dei dati.


## Obiettivo di questa lezione: solo **O = Obtain**

Oggi impariamo esclusivamente a **reperire, scaricare, riconoscere e aprire i dati**. Non li puliremo, non cercheremo relazioni tra variabili e non costruiremo modelli.

### Nelle prossime lezioni

- **Prossima lezione — S:** useremo **pandas** per lo *scrubbing*, cioè per correggere valori mancanti, duplicati, tipi errati e formati incoerenti.
- **Lezione seguente — E:** esploreremo i dati e impareremo a **visualizzarli** con grafici semplici.
- **In seguito — M e N:** costruiremo modelli e, infine, interpreteremo i risultati.



## 1. Prima di scaricare: tre domande

Prima di acquisire un dataset, chiediamoci:

1. **Da dove proviene?** La fonte è affidabile e documentata?
2. **Possiamo usarlo?** Licenza, privacy e autorizzazioni permettono il download?
3. **Dove lo salviamo?** I file grandi non dovrebbero essere inseriti nel repository Git.

In questa lezione useremo una cartella `data` nella home dell'utente. In questo modo il codice funziona su sistemi diversi senza dipendere dal nome di una persona o da un percorso assoluto specifico.


In [1]:
from pathlib import Path

cartella_dati = Path.home() / "data"
cartella_dati.mkdir(exist_ok=True)

print(cartella_dati)


/home/gloria/data


## 2. Ottenere un file dal Web tramite `urllib.request`

https://docs.python.org/3/library/urllib.request.html

Useremo un piccolo dataset pubblico dell'esperimento MAGIC, disponibile nell'[archivio UCI Machine Learning](https://archive.ics.uci.edu/ml/datasets/MAGIC+Gamma+Telescope). Il file `.data` contiene i dati; il file `.names` descrive le colonne e la provenienza.

Scaricare anche la documentazione è importante: numeri senza contesto possono essere interpretati in modo sbagliato.

#### urllib.request.urlretrieve 

La sintassi del comandoche permette di copiare un 'network object' in locale è: 
```python
urllib.request.urlretrieve(url, filename=None, reporthook=None, data=None)
```

Restituisce una tupla `(filename, headers)`, dove *filename* è il nome del file locale in cui è stato salvato l'oggetto, mentre *headers* contiene le informazioni restituite dal metodo `info()` dell'oggetto ottenuto tramite `urlopen()` (nel caso di una risorsa remota).



In [2]:
from urllib.request import urlretrieve

base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/magic/"
nomi_file = ["magic04.data", "magic04.names"]

for nome in nomi_file:
    destinazione = cartella_dati / nome
    if not destinazione.exists():
        urlretrieve(base_url + nome, destinazione)
    print(f"Disponibile: {destinazione}")


Disponibile: /home/gloria/data/magic04.data
Disponibile: /home/gloria/data/magic04.names


### Controllare che il download sia riuscito

Per ora verifichiamo soltanto che i file esistano e osserviamo la loro dimensione. L'analisi del contenuto arriverà nelle lezioni successive.


In [3]:
for nome in nomi_file:
    percorso = cartella_dati / nome
    dimensione_kb = percorso.stat().st_size / 1024
    print(f"{nome}: {dimensione_kb:.1f} KB")


magic04.data: 1442.8 KB
magic04.names: 5.3 KB


### Leggere la documentazione

Un file di testo può essere letto con `read_text`. Mostriamo soltanto le prime righe per mantenere l'output breve e leggibile.


In [4]:
documentazione = cartella_dati / "magic04.names"
righe = documentazione.read_text(encoding="utf-8", errors="replace").splitlines()

for riga in righe[:15]:
    print(riga)


1. Title of Database: MAGIC gamma telescope data 2004

2. Sources:

   (a) Original owner of the database:

       R. K. Bock
       Major Atmospheric Gamma Imaging Cherenkov Telescope project (MAGIC)
       http://wwwmagic.mppmu.mpg.de
       rkb@mail.cern.ch

   (b) Donor:

       P. Savicky
       Institute of Computer Science, AS of CR


## 3. Ottenere dati tramite web scraping 

Il web scraping è una tecnica per estrarre automaticamente informazioni strutturate da pagine web. In generale vengono usate due librerie:
   * `requests`per scaricare il contenuto HTML della pagina; <br>
   * `BeautifulSoup` per analizzare la struttura e selezionare gli elementi di interesse, ad esempio tabelle, titoli, link o testi. <br>
I dati estratti possono poi essere organizzati e analizzati, ad esempio trasformandoli in un DataFrame Pandas.

Il flusso di lavoro può essere:
   * requests scarica la pagina 
   * BeautifulSoup interpreta l’HTML 
   * cerchi nell’HTML gli elementi che ti interessano 
   * estrai i dati.

#### Principali elementi HTML utili nel web scraping

| Cosa cerchi          | Tag HTML tipico | BeautifulSoup              |
| -------------------- | --------------- | -------------------------- |
| Titolo principale    | `<h1>`          | `soup.find("h1")`          |
| Sottotitoli          | `<h2>`, `<h3>`  | `soup.find_all("h2")`      |
| Paragrafi            | `<p>`           | `soup.find_all("p")`       |
| Link                 | `<a>`           | `soup.find_all("a")`       |
| Immagini             | `<img>`         | `soup.find_all("img")`     |
| Tabelle              | `<table>`       | `soup.find_all("table")`   |
| Righe tabella        | `<tr>`          | `table.find_all("tr")`     |
| Celle tabella        | `<td>`          | `row.find_all("td")`       |
| Intestazioni tabella | `<th>`          | `row.find_all("th")`       |
| Liste                | `<ul>`, `<ol>`  | `soup.find_all("ul")`      |
| Elementi di lista    | `<li>`          | `soup.find_all("li")`      |
| Video                | `<video>`       | `soup.find_all("video")`   |
| Audio                | `<audio>`       | `soup.find_all("audio")`   |
| Form                 | `<form>`        | `soup.find_all("form")`    |
| Campi input          | `<input>`       | `soup.find_all("input")`   |
| Bottoni              | `<button>`      | `soup.find_all("button")`  |
| Sezioni generiche    | `<div>`         | `soup.find_all("div")`     |
| Sezioni semantiche   | `<section>`     | `soup.find_all("section")` |

#### Possibili status_code
| Codice    | Significato          |
| --------- | -------------------- |
| `200`     |  OK                 |
| `301/302` |  Redirect          |
| `403`     |  Forbidden          |
| `404`     |  Pagina non trovata |
| `429`     |  Troppe richieste  |
| `500`     |  Errore del server |


In [6]:
# installiamo le librerie necessarie
!pip install requests
!pip install beautifulsoup4

import requests
from bs4 import BeautifulSoup
import pandas as pd


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [7]:
# Scarichiamo con request la pagina wikipedia dell'airbus a380
# contiene immagini, video e tabelle

url = "https://it.wikipedia.org/wiki/Airbus_A380"
headers = {"User-Agent": "MyLearningWebScraper/1.0 (educational project)"} # x wikipedia

response = requests.get(url, headers=headers)

print(response.text)
print(response.status_code)

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-feature-navigation-update-disabled vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="it" dir="ltr">
<head>
<meta charset="UTF-8">
<title>Airbus A380 - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-

In [11]:
# Cerchiamo le tabelle (1 sola)
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table", class_="wikitable")

print(len(tables))

df = pd.read_html(str(tables))[0]
df

1


/tmp/ipykernel_24902/1410516665.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tables))[0]


,Dati,A380-800[4]
0,Codice ICAO,A388
1,Codice IATA,380
2,Equipaggio in cabina di pilotaggio,2 piloti
3,Capacità massima passeggeri,868 (EASA) 853 (FAA)
4,Lunghezza totale,"72,57 m"
5,Diametro cabina,"5,80 m (ponte superiore) 6,50 m (ponte princip..."
6,Diametro fusoliera,"7,14 m"
7,Larghezza piano orizzontale,"30,37 m"
8,Apertura alare,"79,75 m"
9,Superficie alare,"845,0 m²"


In [14]:
# Cerchiamo i video
video = soup.find_all("video")
len(video)

1

In [19]:
# visualize a video with IPython.display
from IPython.display import Video
video_url = soup.find("video").find("source").get("src")
Video(url=video_url)

In [16]:
# Cerchiamo le immagini 
images = soup.find_all("img")
len(images)

45

In [28]:
# visualize an image with IPython.display
from IPython.display import display, HTML
display(HTML("".join(
    f'<img src="https:{img.get("src")}" height="100">'
    for img in images[:10]
    if img.get("src", "").startswith("//")
)))

## 4. Ottenere dati da un computer remoto

Non tutti i dati si trovano su una pagina Web. In un laboratorio, per esempio, possono essere conservati su un server accessibile con credenziali.

Dal terminale si può usare `scp`:

```bash
scp utente@server:/percorso/dataset.csv ~/data/
```

Il comando è solo un modello: bisogna sostituire utente, server e percorso. Non inserire password o chiavi private nel notebook. Se il server si trova dietro una rete protetta, occorre seguire le regole di accesso stabilite dall'organizzazione.


## 5. Riconoscere i formati più comuni

I dati possono essere salvati in molti formati. In questa fase ci basta capire **che tipo di file abbiamo ottenuto** e come aprirlo. La pulizia con pandas sarà affrontata nella prossima lezione.

| Formato | Caratteristica principale | Esempio |
|---|---|---|
| Testo | leggibile direttamente | `.txt` |
| CSV | tabella separata da virgole o altri simboli | `.csv`, `.data` |
| JSON | struttura con chiavi e valori | `.json` |
| Binario | compatto, richiede la conoscenza del formato | `.bin`, `.dat` |
| HDF5 | dati scientifici grandi e gerarchici | `.h5`, `.hdf5` |
| ROOT | eventi di esperimenti di fisica | `.root` |


### 5.1 File di testo

Quando il file è grande, è meglio non caricarlo tutto in memoria. Possiamo leggere poche righe alla volta.


In [32]:
from pathlib import Path

file_path = cartella_dati / "esempio.txt"

# OUTPUT: scriviamo e salviamo un file di testo
with open(file_path, "w") as file:
    file.write("Ciao!\n")
    file.write("Questo è un semplice file di testo.")

# INPUT: riapriamo il file e leggiamo il contenuto
with open(file_path, "r") as file:
    testo = file.read()

print(testo)

Ciao!
Questo è un semplice file di testo.


### 5.2 File CSV

CSV significa *comma-separated values*. Il separatore è spesso una virgola, ma può essere anche un punto e virgola o una tabulazione. Il modulo `csv` della libreria standard legge una riga come una lista di valori.

Qui osserviamo soltanto la prima riga: la conversione dei tipi e la scelta dei nomi delle colonne appartengono alla fase di pulizia.


In [34]:
from pathlib import Path
import pandas as pd
import csv

file_path = cartella_dati / "studenti.csv"

# Creiamo una tabella 
df = pd.DataFrame({
    "nome": ["Anna", "Marco", "Luca"],
    "eta": [18, 20, 19]
})

# OUTPUT: salviamo il DataFrame in un file CSV
df.to_csv(file_path, index=False)

# INPUT: riapriamo il file CSV
df_letto = pd.read_csv(file_path)

df_letto

,nome,eta
0,Anna,18
1,Marco,20
2,Luca,19


### 5.3 File JSON

JSON organizza le informazioni con oggetti, liste, chiavi e valori. È molto comune nelle API Web. Creiamo un esempio minimo e poi lo rileggiamo.


In [7]:
import json

esempio = {
    "esperimento": "MAGIC",
    "file": "magic04.data",
    "scaricato": True,
}

file_json = cartella_dati / "esempio.json"
file_json.write_text(json.dumps(esempio, indent=2), encoding="utf-8")


75

In [8]:
contenuto = json.loads(file_json.read_text(encoding="utf-8"))
print(contenuto)


{'esperimento': 'MAGIC', 'file': 'magic04.data', 'scaricato': True}


### 5.4 File binari

Un file binario non è pensato per essere letto come testo. Per interpretarlo dobbiamo conoscere la struttura con cui è stato scritto: ordine dei byte, dimensione e tipo di ogni valore.

L'esempio seguente salva tre interi e li recupera. Nei dati reali, la struttura deve essere fornita da chi produce il file.


In [35]:
import struct

file_binario = cartella_dati / "esempio.bin"

# OUTPUT: converte i numeri in byte e li salva
file_binario.write_bytes(struct.pack("<3i", 10, 20, 30))

# INPUT: legge i byte e li riconverte in numeri
valori = struct.unpack("<3i", file_binario.read_bytes())
print(valori)


(10, 20, 30)


### 5.5 HDF5 e ROOT

Per dataset scientifici grandi o complessi si usano spesso formati specializzati:

- **HDF5** organizza array e metadati in gruppi gerarchici. In Python si può aprire con `h5py`.
- **ROOT** è uno degli strumenti più convenienti per memorizzare e gestire dataset complessi relativi a esperimenti di fisica in cui vengono registrati degli “eventi”, in particolare nella fisica delle alte energie, nucleare e astrofisica. Permette di utilizzare una struttura annidata, con oggetti dati complessi (classi) e riferimenti tra di essi. In Python si può leggere anche con `uproot`.

Queste librerie non fanno parte della libreria standard. Per mantenere la lezione semplice non le installiamo e non analizziamo questi file oggi. La prima operazione resta la stessa: ottenere il file insieme alla documentazione del suo formato.


## 6. Dati in memoria e dati su disco

Alcuni strumenti permettono di leggere una risorsa direttamente da un indirizzo Web. È comodo, ma un file molto grande può occupare troppa memoria. Salvare prima il file su disco offre inoltre alcuni vantaggi:

- il download non deve essere ripetuto a ogni esecuzione;
- è possibile controllare dimensione, nome e provenienza;
- il lavoro rimane riproducibile anche senza connessione;
- si può verificare l'integrità del file.

Per questo abbiamo scaricato il dataset una volta sola usando il controllo `if not destinazione.exists()`.


## 7. Esercizio sulla fase Obtain

1. Crea una lista casuale di numeri e salvala in un file di testo chiamato `simple_data.txt`.

2. Crea una matrice casuale di dimensione `5x5` e salvala in un file di testo chiamato `data.txt`.

3. Carica con Pandas il file remoto https://www.dropbox.com/s/vkl89yce7xjdq4n/regression_generated.csv?dl=1 e rappresenta tramite scatter plot (vedi lezione numpy) tutte le possibili combinazioni dei seguenti campi:

   * `features_1`
   * `features_2`
   * `features_3`

4. Carica con Pandas il file disponibile al seguente URL: https://www.dropbox.com/s/7u3lm737ogbqsg8/mushrooms_categorized.csv?dl=1. Esplora i dati e visualizzane le informazioni principali.

6. Carica il file `user_data.json`, filtra i dati selezionando le righe in cui il campo `CreditCardType` è uguale a `"American Express"`. Successivamente, salva i dati filtrati in un file CSV.


## Riepilogo

In questa lezione abbiamo:

- visto l'intero percorso OSEMN;
- creato una cartella dedicata ai dati;
- scaricato un dataset e la relativa documentazione;
- controllato i file ottenuti;
- visto il web scraping
- riconosciuto alcuni formati comuni;
- distinto il trasferimento dei dati dalla loro successiva elaborazione.
